# 🦷 Dental Disease Detection - YOLOv8 Training

## Dataset: 10,171 intraoral X-ray images
## Classes: Caries, Crown, Filling, Implant, Periapical-lesion
## Target: 80%+ mAP50

In [ ]:
# Install dependencies
!pip install ultralytics albumentations -q

import os
import shutil
from pathlib import Path
import yaml
import json
from datetime import datetime

print("✅ Dependencies installed")

In [ ]:
# Upload your mega-dataset.zip to Kaggle Datasets first!
# Or mount Google Drive if using Colab

# For Kaggle: dataset should be at /kaggle/input/dental-mega-dataset/
# For Colab: mount drive and update path

DATASET_PATH = "/kaggle/input/dental-mega-dataset"  # Update this!
WORK_DIR = "/kaggle/working/dental-ai"

# Create work directory
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f"{WORK_DIR}/runs", exist_ok=True)

print(f"📁 Dataset: {DATASET_PATH}")
print(f"📁 Work dir: {WORK_DIR}")

In [ ]:
# Copy dataset to working directory
if os.path.exists(DATASET_PATH):
    # Check if it's zipped or already extracted
    if os.path.exists(f"{DATASET_PATH}/data.yaml"):
        # Already extracted
        print("✅ Dataset already extracted")
        dataset_dir = DATASET_PATH
    else:
        # Look for zip file
        zip_files = list(Path(DATASET_PATH).glob("*.zip"))
        if zip_files:
            print(f"📦 Extracting {zip_files[0]}...")
            !unzip -q {zip_files[0]} -d {WORK_DIR}
            dataset_dir = WORK_DIR
        else:
            print("❌ No dataset found. Upload mega-dataset.zip first!")
            dataset_dir = None
else:
    print(f"❌ Dataset not found at {DATASET_PATH}")
    print("Upload mega-dataset.zip to Kaggle Datasets first!")
    dataset_dir = None

if dataset_dir:
    # Verify dataset
    yaml_path = f"{dataset_dir}/data.yaml"
    if os.path.exists(yaml_path):
        with open(yaml_path) as f:
            config = yaml.safe_load(f)
        print(f"\n📊 Dataset config:")
        print(f"  Classes: {config['nc']}")
        print(f"  Names: {config['names']}")
        
        # Count images
        for split in ['train', 'val', 'test']:
            img_dir = f"{dataset_dir}/{split}/images"
            if os.path.exists(img_dir):
                count = len(list(Path(img_dir).glob("*.jpg"))) + len(list(Path(img_dir).glob("*.png")))
                print(f"  {split}: {count} images")

## 🚀 Training Configuration

In [ ]:
# Training configuration
CONFIG = {
    # Model
    "model": "yolov8x.pt",  # Use yolov8x for best accuracy
    
    # Training
    "epochs": 150,           # More epochs for better convergence
    "patience": 30,          # Early stopping
    "batch": 4,              # Limited by GPU memory
    "imgsz": 640,            # Start with 640, can increase later
    
    # Optimizer
    "optimizer": "AdamW",
    "lr0": 0.001,            # Initial learning rate
    "lrf": 0.01,             # Final learning rate (lr0 * lrf)
    "momentum": 0.937,
    "weight_decay": 0.0005,
    
    # Augmentation
    "mosaic": 1.0,           # Mosaic augmentation
    "mixup": 0.15,           # Mixup augmentation
    "copy_paste": 0.1,       # Copy-paste augmentation
    "hsv_h": 0.015,          # HSV-Hue augmentation
    "hsv_s": 0.7,            # HSV-Saturation augmentation
    "hsv_v": 0.4,            # HSV-Value augmentation
    "flipud": 0.0,           # No vertical flip (dental X-rays)
    "fliplr": 0.5,           # Horizontal flip
    "erasing": 0.4,          # Random erasing
    
    # Other
    "device": 0,             # GPU
    "workers": 2,
    "project": f"{WORK_DIR}/runs",
    "name": f"dental_yolov8x_{datetime.now().strftime('%Y%m%d_%H%M')}",
}

print("⚙️ Training configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## 🏋️ Train YOLOv8 Model

In [ ]:
from ultralytics import YOLO
import torch

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"🎮 GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("⚠️ No GPU found! Training will be slow.")

# Initialize model
model = YOLO(CONFIG["model"])
print(f"\n🤖 Model: {CONFIG['model']}")
print(f"Parameters: {sum(p.numel() for p in model.model.parameters()) / 1e6:.1f}M")

In [ ]:
# Start training
print("🏋️ Starting training...")
print(f"⏰ Estimated time: {CONFIG['epochs'] * 8 / 60:.1f} hours (based on ~8 min/epoch)")
print("=" * 60)

results = model.train(
    data=f"{dataset_dir}/data.yaml",
    epochs=CONFIG["epochs"],
    batch=CONFIG["batch"],
    imgsz=CONFIG["imgsz"],
    device=CONFIG["device"],
    optimizer=CONFIG["optimizer"],
    lr0=CONFIG["lr0"],
    lrf=CONFIG["lrf"],
    momentum=CONFIG["momentum"],
    weight_decay=CONFIG["weight_decay"],
    patience=CONFIG["patience"],
    mosaic=CONFIG["mosaic"],
    mixup=CONFIG["mixup"],
    copy_paste=CONFIG["copy_paste"],
    hsv_h=CONFIG["hsv_h"],
    hsv_s=CONFIG["hsv_s"],
    hsv_v=CONFIG["hsv_v"],
    flipud=CONFIG["flipud"],
    fliplr=CONFIG["fliplr"],
    erasing=CONFIG["erasing"],
    workers=CONFIG["workers"],
    project=CONFIG["project"],
    name=CONFIG["name"],
    exist_ok=True,
    pretrained=True,
    verbose=True,
)

print("\n✅ Training complete!")

## 📊 Evaluate Results

In [ ]:
# Print results
print("=" * 60)
print("📊 TRAINING RESULTS")
print("=" * 60)

best_path = Path(CONFIG["project"]) / CONFIG["name"] / "weights" / "best.pt"
last_path = Path(CONFIG["project"]) / CONFIG["name"] / "weights" / "last.pt"

if best_path.exists():
    print(f"\n✅ Best model: {best_path}")
    print(f"   Size: {best_path.stat().st_size / 1024 / 1024:.1f} MB")
    
    # Load and evaluate
    best_model = YOLO(str(best_path))
    metrics = best_model.val(data=f"{dataset_dir}/data.yaml", device=CONFIG["device"])
    
    print(f"\n📈 Metrics:")
    print(f"  mAP50: {metrics.box.map50:.4f} ({metrics.box.map50*100:.1f}%)")
    print(f"  mAP50-95: {metrics.box.map:.4f} ({metrics.box.map*100:.1f}%)")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")
    
    # Per-class metrics
    print(f"\n📊 Per-class mAP50:")
    class_names = ['Caries', 'Crown', 'Filling', 'Implant', 'Periapical-lesion']
    for i, name in enumerate(class_names):
        if i < len(metrics.box.maps):
            print(f"  {name}: {metrics.box.maps[i]:.4f}")
else:
    print("❌ No best.pt found")

## 💾 Export Model

In [ ]:
# Create submission directory
submission_dir = Path("/kaggle/working/submission")
submission_dir.mkdir(exist_ok=True)

# Copy best model
if best_path.exists():
    shutil.copy2(best_path, submission_dir / "best.pt")
    print(f"✅ Copied best.pt to {submission_dir}")
    
    # Copy config
    shutil.copy2(f"{dataset_dir}/data.yaml", submission_dir / "data.yaml")
    
    # Create README
    readme = f"""# Dental Disease Detection Model
    
## Training Info
- Model: {CONFIG['model']}
- Dataset: {dataset_dir}
- Epochs: {CONFIG['epochs']}
- Image size: {CONFIG['imgsz']}
- Batch size: {CONFIG['batch']}
    
## Results
- mAP50: {metrics.box.map50:.4f}
- mAP50-95: {metrics.box.map:.4f}
- Precision: {metrics.box.mp:.4f}
- Recall: {metrics.box.mr:.4f}
    
## Classes
0: Caries
1: Crown
2: Filling
3: Implant
4: Periapical-lesion
    
## Usage
```python
from ultralytics import YOLO
model = YOLO('best.pt')
results = model.predict('xray.jpg')
```
"""
    (submission_dir / "README.md").write_text(readme)
    
    # Create submission zip
!cd /kaggle/working && zip -r submission.zip submission/
print(f"\n📦 Submission ready: /kaggle/working/submission.zip")
else:
    print("❌ No model to export")

## 🔄 Optional: Fine-tune with Higher Resolution

After initial training, fine-tune with imgsz=1280 for better detection of small objects.

In [ ]:
# Fine-tune with higher resolution (optional)
FINE_TUNE = False  # Set to True to enable

if FINE_TUNE and best_path.exists():
    print("🔄 Fine-tuning with imgsz=1280...")
    
    model_ft = YOLO(str(best_path))
    results_ft = model_ft.train(
        data=f"{dataset_dir}/data.yaml",
        epochs=50,           # Fewer epochs for fine-tuning
        batch=2,             # Smaller batch for higher resolution
        imgsz=1280,          # Higher resolution
        device=CONFIG["device"],
        lr0=0.0001,          # Lower learning rate
        patience=15,
        project=CONFIG["project"],
        name=CONFIG["name"] + "_finetune",
        exist_ok=True,
    )
    print("✅ Fine-tuning complete!")
else:
    print("⏭️ Skipping fine-tuning (set FINE_TUNE=True to enable)")